In [1]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ==========================================
# 1. Configuración
# ==========================================

ARCHIVO_DATOS = "reseñas.csv"
ARCHIVO_MODELO = "modelo_sentimientos.pkl"

COLUMNA_TEXTO = "reseña"
COLUMNA_ETIQUETA = "sentimiento"


# ==========================================
# 2. Crear datos de ejemplo
# ==========================================

if not os.path.exists(ARCHIVO_DATOS):
    datos_ejemplo = pd.DataFrame({
        "reseña": [
            "Excelente producto, estoy muy satisfecho",
            "Me encantó la calidad y el diseño",
            "La compra fue rápida y todo llegó perfecto",
            "Muy buen servicio, lo recomiendo totalmente",
            "El producto funciona de maravilla",
            "Estoy feliz con mi compra",
            "La calidad es bastante buena",
            "Es un producto útil y confiable",
            "El producto cumple con lo prometido",
            "La atención fue excelente",
            "El producto es normal, nada especial",
            "Cumple su función, pero podría mejorar",
            "La calidad es aceptable por el precio",
            "No está mal, aunque esperaba algo más",
            "El servicio fue regular",
            "El producto funciona de manera promedio",
            "No me gustó, llegó defectuoso",
            "Pésima calidad, fue una pérdida de dinero",
            "Estoy muy decepcionado con la compra",
            "El producto dejó de funcionar rápidamente",
            "El servicio fue terrible",
            "No recomiendo este producto",
            "La entrega fue tardía y el producto llegó roto",
            "La calidad es muy mala",
            "Una experiencia horrible"
        ],
        "sentimiento": [
            "positiva",
            "positiva",
            "positiva",
            "positiva",
            "positiva",
            "positiva",
            "positiva",
            "positiva",
            "positiva",
            "positiva",
            "neutral",
            "neutral",
            "neutral",
            "neutral",
            "neutral",
            "neutral",
            "negativa",
            "negativa",
            "negativa",
            "negativa",
            "negativa",
            "negativa",
            "negativa",
            "negativa",
            "negativa"
        ]
    })

    datos_ejemplo.to_csv(
        ARCHIVO_DATOS,
        index=False,
        encoding="utf-8"
    )

    print(f"Se creó el archivo de ejemplo: {ARCHIVO_DATOS}")


# ==========================================
# 3. Cargar los datos
# ==========================================

try:
    datos = pd.read_csv(
        ARCHIVO_DATOS,
        encoding="utf-8"
    )
except FileNotFoundError:
    print(f"No se encontró el archivo '{ARCHIVO_DATOS}'.")
    exit()


columnas_requeridas = [
    COLUMNA_TEXTO,
    COLUMNA_ETIQUETA
]

if not all(
    columna in datos.columns
    for columna in columnas_requeridas
):
    print("El archivo CSV debe contener estas columnas:")
    print("reseña, sentimiento")
    exit()


# ==========================================
# 4. Limpiar los datos
# ==========================================

datos = datos.dropna(
    subset=columnas_requeridas
).copy()

datos[COLUMNA_TEXTO] = datos[
    COLUMNA_TEXTO
].astype(str)

datos[COLUMNA_ETIQUETA] = datos[
    COLUMNA_ETIQUETA
].astype(str)

datos = datos[
    datos[COLUMNA_TEXTO].str.strip() != ""
]

if datos[COLUMNA_ETIQUETA].nunique() < 2:
    print("Se necesitan al menos dos tipos de sentimiento.")
    exit()


X = datos[COLUMNA_TEXTO]
y = datos[COLUMNA_ETIQUETA]


# ==========================================
# 5. Separar entrenamiento y prueba
# ==========================================

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = (
    train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )
)


# ==========================================
# 6. Crear el modelo
# ==========================================

modelo = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=1,
            sublinear_tf=True
        )
    ),
    (
        "clasificador",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])


# ==========================================
# 7. Entrenar el modelo
# ==========================================

modelo.fit(
    X_entrenamiento,
    y_entrenamiento
)


# ==========================================
# 8. Evaluar el modelo
# ==========================================

predicciones = modelo.predict(
    X_prueba
)

precision = accuracy_score(
    y_prueba,
    predicciones
)

print("\nRESULTADOS DEL MODELO")
print("=====================")
print(f"Exactitud: {precision:.2%}")

print("\nREPORTE DE CLASIFICACIÓN")
print("========================")
print(
    classification_report(
        y_prueba,
        predicciones,
        zero_division=0
    )
)

clases = sorted(y.unique())

matriz = confusion_matrix(
    y_prueba,
    predicciones,
    labels=clases
)

print("MATRIZ DE CONFUSIÓN")
print("===================")
print(
    pd.DataFrame(
        matriz,
        index=[f"Real {clase}" for clase in clases],
        columns=[f"Predicho {clase}" for clase in clases]
    )
)


# ==========================================
# 9. Guardar resultados de prueba
# ==========================================

resultados = pd.DataFrame({
    "reseña": X_prueba.values,
    "sentimiento_real": y_prueba.values,
    "sentimiento_predicho": predicciones
})

resultados.to_csv(
    "resultados_sentimientos.csv",
    index=False,
    encoding="utf-8"
)


# ==========================================
# 10. Guardar el modelo
# ==========================================

joblib.dump(
    modelo,
    ARCHIVO_MODELO
)

print(f"\nModelo guardado como: {ARCHIVO_MODELO}")


# ==========================================
# 11. Analizar nuevas reseñas
# ==========================================

print("\nANÁLISIS DE SENTIMIENTOS")
print("========================")
print("Escribe una reseña para analizarla.")
print("Escribe 'salir' para terminar.\n")

while True:
    reseña_nueva = input("Reseña: ").strip()

    if reseña_nueva.lower() == "salir":
        print("Programa finalizado.")
        break

    if not reseña_nueva:
        print("Debes escribir una reseña.\n")
        continue

    sentimiento = modelo.predict(
        [reseña_nueva]
    )[0]

    probabilidades = modelo.predict_proba(
        [reseña_nueva]
    )[0]

    nombres_clases = modelo.named_steps[
        "clasificador"
    ].classes_

    probabilidades_clases = dict(
        zip(nombres_clases, probabilidades)
    )

    print("\nRESULTADO")
    print("--------")
    print(f"Sentimiento detectado: {sentimiento}")

    print("\nProbabilidades:")
    for clase, probabilidad in (
        probabilidades_clases.items()
    ):
        print(f"{clase}: {probabilidad:.2%}")

    print()

Se creó el archivo de ejemplo: reseñas.csv

RESULTADOS DEL MODELO
Exactitud: 14.29%

REPORTE DE CLASIFICACIÓN
              precision    recall  f1-score   support

    negativa       0.00      0.00      0.00         2
     neutral       0.00      0.00      0.00         2
    positiva       0.20      0.33      0.25         3

    accuracy                           0.14         7
   macro avg       0.07      0.11      0.08         7
weighted avg       0.09      0.14      0.11         7

MATRIZ DE CONFUSIÓN
               Predicho negativa  Predicho neutral  Predicho positiva
Real negativa                  0                 0                  2
Real neutral                   0                 0                  2
Real positiva                  2                 0                  1

Modelo guardado como: modelo_sentimientos.pkl

ANÁLISIS DE SENTIMIENTOS
Escribe una reseña para analizarla.
Escribe 'salir' para terminar.

Reseña: esto me parece cool pero es raro

RESULTADO
--------
Sentimi